In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns
from sentence_transformers import SentenceTransformer
import nltk
from nltk.tokenize import sent_tokenize
import json
import os
from tqdm import tqdm

In [2]:
# Load pretrained Sentence Transformer model and tokenizer

# Choosing all-MiniLM-L6-v2 as it is the smallest model that still performs well
# Other option is all-mpnet-base-v2 which performs slightly better but is much larger and slower
model = SentenceTransformer("all-MiniLM-L6-v2")  

tokenizer = nltk.data.load('tokenizers/punkt/english.pickle') # tokenizer for splitting into sentences

# Create sentence embeddings

## Individual encoding

In [3]:
# Config

data_paths = {
    "Small": [
        "/Users/maxschaffelder/Desktop/Thesis/Data/exp_1/data/small/raw/generation_results_8b_single_v6.jsonl",
        "/Users/maxschaffelder/Desktop/Thesis/Data/exp_1/self_bleu_3/Small/generation_results_8b_multi_v1_self_bleu.jsonl",
        "/Users/maxschaffelder/Desktop/Thesis/Data/exp_1/self_bleu_3/Small/generation_results_8b_human_v1_self_bleu.jsonl",
        "/Users/maxschaffelder/Desktop/Thesis/Data/exp_1/self_bleu_3/Small/dolly_test_Llama_self_bleu.jsonl"
    ]
}

sample_size = 7000

In [8]:
all_human_data = []
all_model_data = []
all_human_sentences = []
all_model_sentences = []

# Load the data
with open(file_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        response_key_human = "response_human"
        response_key_model = [k for k in data.keys() if k.startswith('response_')][1]
        response_human = data[response_key_human]
        response_model = data[response_key_model]
        
        all_human_data.append(response_human)
        all_model_data.append(response_model)


# Tokenize all responses
for response_human, response_model in zip(all_human_data, all_model_data):
    response_human_sentences = tokenizer.tokenize(response_human)
    response_model_sentences = tokenizer.tokenize(response_model)
    all_human_sentences.extend(response_human_sentences)
    all_model_sentences.extend(response_model_sentences)


# Sample 5000 instances from each dataset using the same random indices
# Set a seed for reproducibility
np.random.seed(42)

# Sample the sentences using the same indices
random_indices_human = np.random.choice(len(all_human_sentences), size=sample_size, replace=False)
random_indices_model = np.random.choice(len(all_model_sentences), size=sample_size, replace=False)

sampled_human_sentences = [all_human_sentences[i] for i in random_indices_human]
sampled_model_sentences = [all_model_sentences[i] for i in random_indices_model]

print(f"Number of sampled sentences: {len(sampled_human_sentences)}")



# 2. Calculate embeddings by calling model.encode()
embeddings_human = model.encode(sampled_human_sentences)
embeddings_model = model.encode(sampled_model_sentences)
print(embeddings_human.shape)
print(embeddings_model.shape)



Number of sampled sentences: 7000
(7000, 384)
(7000, 384)


In [10]:
# Create output directories
output_base_dir = "../../Data/analysis/sentence_embeddings/Large/dolly_train_2_Gemini_with_system_prompt_embeddings.jsonl"
file_path = "../../Data/Finetuning/Augmented/Large/Other/dolly_train_2_Gemini_with_system_prompt.jsonl"


# Initialize data containers
all_model_data = []
all_model_sentences = []


# Load the data
with open(file_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        response_key_model = [k for k in data.keys() if k.startswith('response_')][1]
        response_model = data[response_key_model]
        all_model_data.append(response_model)

# Tokenize all responses
for response_model in all_model_data:
    response_model_sentences = sent_tokenize(response_model)
    valid_sentences = [s for s in response_model_sentences if is_valid_sentence(s)]
    all_model_sentences.extend(valid_sentences)
    

# Sample sentences
if len(all_model_sentences) >= sample_size:
    random_indices_model = np.random.choice(len(all_model_sentences), size=sample_size, replace=False)
    sampled_model_sentences = [all_model_sentences[i] for i in random_indices_model]
else:
    sampled_model_sentences = all_model_sentences


# Calculate embeddings
print(f"  Calculating embeddings...")
embeddings_model = model.encode(sampled_model_sentences)

# Save model embeddings
model_output_file = output_base_dir
with open(model_output_file, 'w') as f:
    for sentence, embedding in zip(sampled_model_sentences, embeddings_model):
        entry = {
            "sentence": sentence,
            "embedding": embedding.tolist(),
            "source": model_name
        }
        f.write(json.dumps(entry) + '\n')

print(f"  Saved embeddings to {model_output_file}")



  Calculating embeddings...
  Saved embeddings to ../../Data/analysis/sentence_embeddings/Large/dolly_train_2_Gemini_with_system_prompt_embeddings.jsonl


### Individual human encoding

In [ ]:
## Embedding human sentences
human_files = []
all_human_data = dict()
all_human_sentences = dict()
embeddings_human = dict()
embeddings_human_2d = dict()

for i in range(1, 5):
    file_path = f"../../Data/Finetuning/Dolly/dolly_train_{i}.jsonl"
    human_files.append(file_path)
    all_human_data[f"dolly_train_{i}"] = []
    all_human_sentences[f"dolly_train_{i}"] = []
    embeddings_human[f"dolly_train_{i}"] = []
    embeddings_human_2d[f"dolly_train_{i}"] = []

for i, file_path in enumerate(human_files):
    # Load the data
    with open(file_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            response_key_human = "response_human"
            response_human = data[response_key_human]
            all_human_data[f"dolly_train_{i+1}"].append(response_human)

            response_human_sentences = tokenizer.tokenize(response_human)
            all_human_sentences[f"dolly_train_{i+1}"].extend(response_human_sentences)
        



# Sample 5000 instances from each dataset using the same random indices
# Set a seed for reproducibility
np.random.seed(42)

# Sample the sentences using the same indices
random_indices_human = np.random.choice(11770, size=5000, replace=False) # 11770 is the number of sentences in dolly_train_3 (smallest dataset)


sampled_human_sentences = dict()
sampled_human_embeddings = dict()

for i in range(1, 5):   
    sampled_human_sentences[f"dolly_train_{i}"] = [all_human_sentences[f"dolly_train_{i}"][j] for j in random_indices_human]
    sampled_human_embeddings[f"dolly_train_{i}"] = model.encode(sampled_human_sentences[f"dolly_train_{i}"])
    


# Batch encoding

In [4]:
def is_valid_sentence(sentence):
    """
    Check if a sentence is valid (not just a number or numbered list marker)
    Returns True if sentence is valid, False otherwise
    """
    # Strip whitespace
    sentence = sentence.strip()
    
    # Check if sentence is empty
    if not sentence:
        return False
    
    # Remove common markdown formatting characters
    cleaned = sentence.replace('#', '').replace('*', '').strip()
        
    # Check if sentence is just a number followed by a period (like "1.", "2.", "### 8.", "**2.", etc)
    if cleaned.replace('.', '').isdigit():
        return False
        
    # Check if sentence is too short (less than 3 characters)
    if len(cleaned) < 3:
        return False
        
    # Check if sentence starts with a number and period but has no other content
    if cleaned.split('.')[0].isdigit() and len(cleaned.split('.')) <= 2:
        return False
    
    return True

### Model data encoding

In [6]:
# Config

data_paths = {
    "Small": [
        "/Users/maxschaffelder/Desktop/Thesis/Data/exp_1/data/small/raw/generation_results_8b_single_v6.jsonl",
        "/Users/maxschaffelder/Desktop/Thesis/Data/exp_1/data/small/raw/generation_results_8b_multi_v1.jsonl",
        "/Users/maxschaffelder/Desktop/Thesis/Data/exp_1/data/small/raw/generation_results_8b_human_v1.jsonl",
        "/Users/maxschaffelder/Desktop/Thesis/Data/exp_1/data/small/raw/dolly_test_Llama.jsonl"
    ]
}


# Create output directories
output_base_dir = "/Users/maxschaffelder/Desktop/Thesis/Data/exp_1/data/sentence_embeddings"

for size in ["Small"]:
    os.makedirs(os.path.join(output_base_dir, size), exist_ok=True)

sample_size = 7000
np.random.seed(42)  # Set seed for reproducibility

# Process each dataset
for size, file_paths in data_paths.items():
    print(f"Processing {size} models...")
    
    for file_path in tqdm(file_paths):
        # Extract model name from file path
        file_name = os.path.basename(file_path)
        model_name = file_name.split('.')[0]
        
        # Initialize data containers
        all_model_data = []
        all_model_sentences = []
        
        # Load the data
        with open(file_path, 'r') as f:
            for line in f:
                data = json.loads(line)
                response_key_model = "response_model"
                response_model = data[response_key_model]
                all_model_data.append(response_model)
        
        # Tokenize all responses
        for response_model in all_model_data:
            # response_model_sentences = tokenizer.tokenize(response_model)
            response_model_sentences = sent_tokenize(response_model)
            valid_sentences = [s for s in response_model_sentences if is_valid_sentence(s)]
            all_model_sentences.extend(valid_sentences)
            
            # all_model_sentences.extend(response_model_sentences)
        
        # Sample sentences
        if len(all_model_sentences) >= sample_size:
            random_indices_model = np.random.choice(len(all_model_sentences), size=sample_size, replace=False)
            sampled_model_sentences = [all_model_sentences[i] for i in random_indices_model]
        else:
            print(f"Warning: Not enough model sentences in {file_name}. Using all available.")
            sampled_model_sentences = all_model_sentences
        
        print(f"  {file_name}: {len(sampled_model_sentences)} model sentences")
        
        # Calculate embeddings
        print(f"  Calculating embeddings for {file_name}...")
        embeddings_model = model.encode(sampled_model_sentences)
        
        # Save model embeddings
        model_output_file = os.path.join(output_base_dir, size, f"{model_name}_embeddings.jsonl")
        with open(model_output_file, 'w') as f:
            for sentence, embedding in zip(sampled_model_sentences, embeddings_model):
                entry = {
                    "sentence": sentence,
                    "embedding": embedding.tolist(),
                    "source": model_name
                }
                f.write(json.dumps(entry) + '\n')
        
        print(f"  Saved embeddings to {model_output_file}")

print("All embeddings have been generated and saved.")



Processing Small models...


  0%|          | 0/4 [00:00<?, ?it/s]

  generation_results_8b_single_v6.jsonl: 7000 model sentences
  Calculating embeddings for generation_results_8b_single_v6.jsonl...


 25%|██▌       | 1/4 [00:09<00:28,  9.37s/it]

  Saved embeddings to /Users/maxschaffelder/Desktop/Thesis/Data/exp_1/data/sentence_embeddings/Small/generation_results_8b_single_v6_embeddings.jsonl
  generation_results_8b_multi_v1.jsonl: 7000 model sentences
  Calculating embeddings for generation_results_8b_multi_v1.jsonl...


...

  Saved embeddings to /Users/maxschaffelder/Desktop/Thesis/Data/exp_1/data/sentence_embeddings/Small/generation_results_8b_multi_v1_embeddings.jsonl
  generation_results_8b_human_v1.jsonl: 7000 model sentences
  Calculating embeddings for generation_results_8b_human_v1.jsonl...


 75%|███████▌  | 3/4 [00:30<00:10, 10.21s/it]

  Saved embeddings to /Users/maxschaffelder/Desktop/Thesis/Data/exp_1/data/sentence_embeddings/Small/generation_results_8b_human_v1_embeddings.jsonl
  dolly_test_Llama.jsonl: 7000 model sentences
  Calculating embeddings for dolly_test_Llama.jsonl...


100%|██████████| 4/4 [00:40<00:00, 10.09s/it]

  Saved embeddings to /Users/maxschaffelder/Desktop/Thesis/Data/exp_1/data/sentence_embeddings/Small/dolly_test_Llama_embeddings.jsonl
All embeddings have been generated and saved.


### Human data encoding

In [6]:
# Config

data_paths_human = [
    "../../Data/Finetuning/Dolly/dolly_train_1.jsonl",
    "../../Data/Finetuning/Dolly/dolly_train_2.jsonl",
    "../../Data/Finetuning/Dolly/dolly_train_3.jsonl",
    "../../Data/Finetuning/Dolly/dolly_train_4.jsonl"
]

# Create output directories
output_base_dir = "../../Data/analysis/sentence_embeddings"
os.makedirs(os.path.join(output_base_dir, "Human"), exist_ok=True)

sample_size = 7000
np.random.seed(42)  # Set seed for reproducibility

# Process each human dataset
print("Processing human datasets...")

for file_path in tqdm(data_paths_human):
    # Extract dataset name from file path
    file_name = os.path.basename(file_path)
    dataset_name = file_name.split('.')[0]
    
    # Initialize data containers
    all_human_data = []
    all_human_sentences = []
    
    # Load the data
    with open(file_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            # Get the human response (typically the first response key)
            response_key_human = [k for k in data.keys() if k.startswith('response_human')][0]
            response_human = data[response_key_human]
            all_human_data.append(response_human)
    
    # Tokenize all responses
    for response_human in all_human_data:
        response_human_sentences = tokenizer.tokenize(response_human)
        valid_sentences = [s for s in response_human_sentences if is_valid_sentence(s)]
        all_human_sentences.extend(valid_sentences)
        # all_human_sentences.extend(response_human_sentences)
    
    # Sample sentences
    if len(all_human_sentences) >= sample_size:
        random_indices_human = np.random.choice(len(all_human_sentences), size=sample_size, replace=False)
        sampled_human_sentences = [all_human_sentences[i] for i in random_indices_human]
    else:
        print(f"Warning: Not enough human sentences in {file_name}. Using all available.")
        sampled_human_sentences = all_human_sentences
    
    print(f"  {file_name}: {len(sampled_human_sentences)} human sentences")
    
    # Calculate embeddings
    print(f"  Calculating embeddings for {file_name}...")
    embeddings_human = model.encode(sampled_human_sentences)
    
    # Save human embeddings
    human_output_file = os.path.join(output_base_dir, "Human", f"{dataset_name}_human_embeddings.jsonl")
    with open(human_output_file, 'w') as f:
        for sentence, embedding in zip(sampled_human_sentences, embeddings_human):
            entry = {
                "sentence": sentence,
                "embedding": embedding.tolist(),
                "source": f"{dataset_name}_human"
            }
            f.write(json.dumps(entry) + '\n')
    
    print(f"  Saved embeddings to {human_output_file}")

print("All human embeddings have been generated and saved.")


Processing human datasets...


  0%|          | 0/4 [00:00<?, ?it/s]

  dolly_train_1.jsonl: 7000 human sentences
  Calculating embeddings for dolly_train_1.jsonl...


 25%|██▌       | 1/4 [00:09<00:29,  9.90s/it]

  Saved embeddings to ../../Data/analysis/sentence_embeddings/Human/dolly_train_1_human_embeddings.jsonl
  dolly_train_2.jsonl: 7000 human sentences
  Calculating embeddings for dolly_train_2.jsonl...


...

  Saved embeddings to ../../Data/analysis/sentence_embeddings/Human/dolly_train_2_human_embeddings.jsonl
  dolly_train_3.jsonl: 7000 human sentences
  Calculating embeddings for dolly_train_3.jsonl...


 75%|███████▌  | 3/4 [00:27<00:08,  8.94s/it]

  Saved embeddings to ../../Data/analysis/sentence_embeddings/Human/dolly_train_3_human_embeddings.jsonl
  dolly_train_4.jsonl: 7000 human sentences
  Calculating embeddings for dolly_train_4.jsonl...


100%|██████████| 4/4 [00:36<00:00,  9.01s/it]

  Saved embeddings to ../../Data/analysis/sentence_embeddings/Human/dolly_train_4_human_embeddings.jsonl
All human embeddings have been generated and saved.
